In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
df = pd.read_csv("urlset.csv")

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

# Remove rows with missing labels
df = df.dropna(subset=["label"])

print("Dataset after cleaning:", df.shape)

In [ ]:
print("\nURL Class Distribution:")
print(df["label"].value_counts())

In [ ]:
plt.figure(figsize=(6, 5))

class_counts = df["label"].value_counts().sort_index()

plt.bar(
    ["Legitimate", "Phishing"],
    class_counts
)

plt.title("URL Class Distribution")
plt.xlabel("URL Class")
plt.ylabel("Number of URLs")

plt.tight_layout()
plt.show()

In [ ]:
# Remove URL column because the model will use extracted features
X = df.drop(columns=["label", "domain"])

# Target
y = df["label"]

print("Number of features:", X.shape[1])

In [ ]:
X = X.select_dtypes(include=[np.number])

print("Numerical features:", X.shape[1])

In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print("========== MODEL PERFORMANCE ==========")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

In [ ]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Legitimate",
            "Phishing"
        ],
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("Logistic Regression - Confusion Matrix")

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    [0, 1],
    ["Legitimate", "Phishing"]
)

plt.yticks(
    [0, 1],
    ["Legitimate", "Phishing"]
)

plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
# Get feature coefficients

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

# Calculate absolute importance
coefficients["Importance"] = abs(
    coefficients["Coefficient"]
)

# Sort by importance
coefficients = coefficients.sort_values(
    by="Importance",
    ascending=False
)

# Top 15 features
top_features = coefficients.head(15)

print("\nTop 15 Important Features:")
print(top_features)

In [ ]:
plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)

plt.title(
    "Top 15 Logistic Regression Feature Importance"
)

plt.xlabel("Absolute Coefficient Value")
plt.ylabel("URL Features")

plt.tight_layout()
plt.show()